# 02 - Ingesta Bronze

En este notebook se cargan los archivos originales del proyecto y se almacenan como tablas Delta en la capa Bronze.

La capa Bronze conserva la estructura original de los archivos, agrega trazabilidad de carga y registra el nombre del archivo fuente.

In [0]:
from pyspark.sql.functions import current_timestamp, lit

In [0]:
# Configuración general del proyecto

catalog_name = "anemia_ayacucho"

schema_landing = "landing"
schema_bronze = "bronze"

volume_name = "fuentes"

path_base = f"/Volumes/{catalog_name}/{schema_landing}/{volume_name}/input"

spark.sql(f"USE CATALOG {catalog_name}")

print("Ruta base:", path_base)

Ruta base: /Volumes/anemia_ayacucho/landing/fuentes/input


In [0]:
# Validación de archivos disponibles

display(dbutils.fs.ls(path_base))

path,name,size,modificationTime
dbfs:/Volumes/anemia_ayacucho/landing/fuentes/input/data_2023.csv,data_2023.csv,24631,1782454010000
dbfs:/Volumes/anemia_ayacucho/landing/fuentes/input/data_2024.csv,data_2024.csv,24598,1782454011000
dbfs:/Volumes/anemia_ayacucho/landing/fuentes/input/data_2025.csv,data_2025.csv,24514,1782454010000
dbfs:/Volumes/anemia_ayacucho/landing/fuentes/input/maestro_zonas.json,maestro_zonas.json,3056,1782454010000


In [0]:
# Función para cargar archivos CSV a la capa Bronze

def cargar_csv_bronze(nombre_archivo, nombre_tabla):
    ruta_archivo = f"{path_base}/{nombre_archivo}"
    tabla_destino = f"{catalog_name}.{schema_bronze}.{nombre_tabla}"
    
    df = (
        spark.read
        .option("header", "true")
        .option("sep", ";")
        .option("inferSchema", "false")
        .csv(ruta_archivo)
    )
    
    df_bronze = (
        df.withColumn("fecha_carga", current_timestamp())
          .withColumn("archivo_fuente", lit(nombre_archivo))
    )
    
    (
        df_bronze.write
        .format("delta")
        .option("overwriteSchema", "true")
        .option("delta.columnMapping.mode", "name")
        .mode("overwrite")
        .saveAsTable(tabla_destino)
    )
    
    print(f"Tabla creada correctamente: {tabla_destino}")
    print(f"Registros cargados: {df_bronze.count()}")
    
    return df_bronze

In [0]:
# Carga de archivos CSV anuales a Bronze

df_bronze_2023 = cargar_csv_bronze("data_2023.csv", "anemia_2023")
df_bronze_2024 = cargar_csv_bronze("data_2024.csv", "anemia_2024")
df_bronze_2025 = cargar_csv_bronze("data_2025.csv", "anemia_2025")

Tabla creada correctamente: anemia_ayacucho.bronze.anemia_2023
Registros cargados: 549
Tabla creada correctamente: anemia_ayacucho.bronze.anemia_2024
Registros cargados: 549
Tabla creada correctamente: anemia_ayacucho.bronze.anemia_2025
Registros cargados: 549


In [0]:
# Vista previa de la tabla Bronze 2023

display(spark.table(f"{catalog_name}.{schema_bronze}.anemia_2023").limit(10))

UBIGEO,PROVINCIA,DISTRITO,N° EVALUADOS,N° NIÑOS CON ANEMIA,AÑO,ZONAS,fecha_carga,archivo_fuente
1,Cangallo,Cangallo,21,6,2023,Zona Norte,2026-06-26T06:28:20.042Z,data_2023.csv
1,Cangallo,Cangallo,21,6,2023,Zona Sur,2026-06-26T06:28:20.042Z,data_2023.csv
1,Cangallo,Cangallo,21,6,2023,Zona Este,2026-06-26T06:28:20.042Z,data_2023.csv
1,Cangallo,Cangallo,21,6,2023,Zona Oeste,2026-06-26T06:28:20.042Z,data_2023.csv
1,Cangallo,Cangallo,21,6,2023,Zona Centro,2026-06-26T06:28:20.042Z,data_2023.csv
1,Cangallo,Cangallo,21,6,2023,Zona Noreste,2026-06-26T06:28:20.042Z,data_2023.csv
1,Cangallo,Cangallo,20,5,2023,Zona Noroeste,2026-06-26T06:28:20.042Z,data_2023.csv
1,Cangallo,Cangallo,20,5,2023,Zona Sureste,2026-06-26T06:28:20.042Z,data_2023.csv
1,Cangallo,Cangallo,20,5,2023,Zona Suroeste,2026-06-26T06:28:20.042Z,data_2023.csv
2,Cangallo,Chuschi,26,11,2023,Zona Norte,2026-06-26T06:28:20.042Z,data_2023.csv


In [0]:
# Lectura del archivo JSON maestro de zonas

ruta_json = f"{path_base}/maestro_zonas.json"

df_zonas = (
    spark.read
    .option("multiline", "true")
    .json(ruta_json)
)

df_zonas_bronze = (
    df_zonas.withColumn("fecha_carga", current_timestamp())
            .withColumn("archivo_fuente", lit("maestro_zonas.json"))
)

display(df_zonas_bronze)

abreviatura,descripcion,id_zona,orden_visualizacion,orientacion,tipo_zona,usar_en_dashboard,zona,fecha_carga,archivo_fuente
N,Clasificación territorial correspondiente al sector norte del ámbito analizado.,1,1,Norte,Cardinal,true,Zona Norte,2026-06-26T06:28:46.212Z,maestro_zonas.json
S,Clasificación territorial correspondiente al sector sur del ámbito analizado.,2,2,Sur,Cardinal,true,Zona Sur,2026-06-26T06:28:46.212Z,maestro_zonas.json
E,Clasificación territorial correspondiente al sector este del ámbito analizado.,3,3,Este,Cardinal,true,Zona Este,2026-06-26T06:28:46.212Z,maestro_zonas.json
O,Clasificación territorial correspondiente al sector oeste del ámbito analizado.,4,4,Oeste,Cardinal,true,Zona Oeste,2026-06-26T06:28:46.212Z,maestro_zonas.json
C,Clasificación territorial correspondiente al sector central del ámbito analizado.,5,5,Centro,Central,true,Zona Centro,2026-06-26T06:28:46.212Z,maestro_zonas.json
NE,Clasificación territorial correspondiente al sector noreste del ámbito analizado.,6,6,Noreste,Intercardinal,true,Zona Noreste,2026-06-26T06:28:46.212Z,maestro_zonas.json
NO,Clasificación territorial correspondiente al sector noroeste del ámbito analizado.,7,7,Noroeste,Intercardinal,true,Zona Noroeste,2026-06-26T06:28:46.212Z,maestro_zonas.json
SE,Clasificación territorial correspondiente al sector sureste del ámbito analizado.,8,8,Sureste,Intercardinal,true,Zona Sureste,2026-06-26T06:28:46.212Z,maestro_zonas.json
SO,Clasificación territorial correspondiente al sector suroeste del ámbito analizado.,9,9,Suroeste,Intercardinal,true,Zona Suroeste,2026-06-26T06:28:46.212Z,maestro_zonas.json


In [0]:
# Escritura del JSON maestro de zonas en formato Delta

tabla_zonas_bronze = f"{catalog_name}.{schema_bronze}.maestro_zonas"

(
    df_zonas_bronze.write
    .format("delta")
    .option("overwriteSchema", "true")
    .mode("overwrite")
    .saveAsTable(tabla_zonas_bronze)
)

print(f"Tabla creada correctamente: {tabla_zonas_bronze}")
print(f"Registros cargados: {df_zonas_bronze.count()}")

Tabla creada correctamente: anemia_ayacucho.bronze.maestro_zonas
Registros cargados: 9


In [0]:
# Validación de tablas creadas en Bronze

display(spark.sql(f"SHOW TABLES IN {catalog_name}.{schema_bronze}"))

database,tableName,isTemporary
bronze,anemia_2023,false
bronze,anemia_2024,false
bronze,anemia_2025,false
bronze,maestro_zonas,false


In [0]:
# Conteo de registros por tabla Bronze

conteos_bronze = [
    ("anemia_2023", spark.table(f"{catalog_name}.{schema_bronze}.anemia_2023").count()),
    ("anemia_2024", spark.table(f"{catalog_name}.{schema_bronze}.anemia_2024").count()),
    ("anemia_2025", spark.table(f"{catalog_name}.{schema_bronze}.anemia_2025").count()),
    ("maestro_zonas", spark.table(f"{catalog_name}.{schema_bronze}.maestro_zonas").count())
]

df_conteos_bronze = spark.createDataFrame(conteos_bronze, ["tabla_bronze", "cantidad_registros"])

display(df_conteos_bronze)

tabla_bronze,cantidad_registros
anemia_2023,549
anemia_2024,549
anemia_2025,549
maestro_zonas,9


## Evidencia esperada

En este notebook se debe capturar:

- Archivos disponibles en el volumen.
- Vista previa de los datos cargados.
- Tablas creadas en Bronze.
- Conteo de registros por tabla.
- Columnas de trazabilidad: `fecha_carga` y `archivo_fuente`.